# LexIQ — Data Preprocessing


## 1. Load Data & Initialize Tokenizer


In [21]:
import pandas as pd
from transformers import AutoTokenizer

data = pd.read_parquet("../data/processed/df_flat.parquet")


In [22]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [23]:
tokenizer.vocab_size

30522

In [24]:
data[data['is_impossible'] == False][['question', 'answer_text']].iloc[0]

question       Highlight the parts (if any) of this contract ...
answer_text                                DISTRIBUTOR AGREEMENT
Name: 0, dtype: str

## 2. Load Contract Texts
We load full contract texts from raw TXT files and merge them with the QA pairs by contract title.

In [25]:
from pathlib import Path

contracts_folder = Path('../data/raw/full_contract_txt')

contracts = [
    {'title' :  f.stem, 'text': f.read_text(encoding='utf8')} for f in contracts_folder.glob('*.txt')
]

contracts = pd.DataFrame(contracts)

df = data.merge(
    contracts,
    on='title',
    how='left',

)

In [26]:
df.shape

(20910, 7)

## 3. Tokenization — Single Example
We test the tokenizer on a single example to verify offset mapping behavior before processing the full dataset.

In [ ]:
example_row = row = df[df['is_impossible'] == False].iloc[0]

sample = tokenizer(example_row['question'],
                   example_row['text'],
                   max_length=512,
                   return_offsets_mapping=True)
print(sample.keys())
print(len(sample["input_ids"]))

In [ ]:
sample['offset_mapping'][:10]


## 4. Span Mapping Logic
DistilBERT operates on tokens, not characters. We use offset_mapping and sequence_ids() to locate the answer span within the context (sequence 1), ignoring question tokens (sequence 0)

In [34]:
for i, j in zip(sample['offset_mapping'], sample.sequence_ids()):
    if i[0] <= 44 and i[1] > 44 and j == 1:
        print(i[0], i[1], j)


44 55 1


## 5. Tokenize & Find Positions — Full Dataset
We apply tokenize_and_find_positions() across all 20,910 rows. For impossible examples, positions default to (0, 0). For truncated answers beyond 512 tokens, positions also fall back to (0, 0).

In [36]:
def tokenize_and_find_positions(row):
    encoding = tokenizer(
        row['question'],
        row['text'],
        max_length=512,
        truncation=True,
        return_offsets_mapping=True
    )

    if row['is_impossible']:
        start_position = 0
        end_position = 0
    else:
        start_position = 0
        end_position = 0
        answer_start = row['answer_start']
        answer_end = answer_start + len(row['answer_text'])

        for idx, (offsets, seq_id) in enumerate(zip(encoding['offset_mapping'], encoding.sequence_ids())):
            if seq_id != 1:
                continue
            if offsets[0] <= answer_start < offsets[1]:
                start_position = idx
            if offsets[0] < answer_end <= offsets[1]:
                end_position = idx

    return {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'start_position': start_position,
        'end_position': end_position
    }

In [40]:
result = tokenize_and_find_positions(df[df['is_impossible'] == False].iloc[0])
print(result['start_position'], result['end_position'])

37 38


In [41]:
df['processed'] = df.apply(tokenize_and_find_positions, axis=1)

In [42]:
df.head()

,title,clause_type,question,is_impossible,answer_text,answer_start,text,processed
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,Highlight the parts (if any) of this contract ...,False,DISTRIBUTOR AGREEMENT,44.0,EXHIBIT 10.6\n\n ...,"{'input_ids': [101, 12944, 1996, 3033, 1006, 2..."
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Highlight the parts (if any) of this contract ...,False,Distributor,244.0,EXHIBIT 10.6\n\n ...,"{'input_ids': [101, 12944, 1996, 3033, 1006, 2..."
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Agreement Date,Highlight the parts (if any) of this contract ...,False,"7th day of September, 1999.",263.0,EXHIBIT 10.6\n\n ...,"{'input_ids': [101, 12944, 1996, 3033, 1006, 2..."
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Effective Date,Highlight the parts (if any) of this contract ...,False,The term of this Agreement shall be ten (10)...,5268.0,EXHIBIT 10.6\n\n ...,"{'input_ids': [101, 12944, 1996, 3033, 1006, 2..."
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Expiration Date,Highlight the parts (if any) of this contract ...,False,The term of this Agreement shall be ten (10)...,5268.0,EXHIBIT 10.6\n\n ...,"{'input_ids': [101, 12944, 1996, 3033, 1006, 2..."


## 6. Save Processed Dataset
Tokenized dataset saved to data/processed/df_processed.parquet for use in fine-tuning notebook (Colab).

In [43]:
df[['title', 'clause_type', 'question', 'is_impossible', 'processed']].to_parquet('../data/processed/df_processed.parquet', index=False)